# SPARKS BASIC

In [1]:
pip install pyspark


#Step 1 - Creating Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Superstore_Assignment") \
    .getOrCreate()

#Step 2 : load Dataset



In [3]:
df = spark.read.csv(
    "/content/Main_dataset/Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

#Step 3 : Exploring dataset


In [4]:
# Total rows

print("Total Rows:", df.count())

# Columns

print(df.columns)

# Schema

df.printSchema()

Total Rows: 9994
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (null

#Step 4 : Data Cleaning

In [6]:
df = df.dropDuplicates()

print("Rows after removing duplicates:")
print(df.count())

Rows after removing duplicates:
9994


#Checking NUll Values

In [7]:
from pyspark.sql.functions import col,isnan,when,count

df.select([
    count(
        when(
            col(c).isNull(),
            c
        )
    ).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



#Filling Missing Values

In [8]:
df = df.fillna({
    "Postal Code":0
})

#Step 5 : Filtering

In [9]:
south_df = df.filter(
    col("Region") == "South"
)

south_df.show()

+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+--------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|       Customer Name|    Segment|      Country|         City|         State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+--------------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|   425|CA-2017-155705| 8/21/2017| 8/23/2017|  Second Class|   NF-18385|    Natalie Fritzler|   Consumer|United States|      Jackson|   Mississippi|      39212| South|FUR-CH-10000015|      F

Furniture Category

In [10]:
furniture_df = df.filter(
    col("Category") == "Furniture"
)

furniture_df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+--------------+--------------+-----------+-------+---------------+---------+------------+--------------------+-------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|          City|         State|Postal Code| Region|     Product ID| Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+--------------+--------------+-----------+-------+---------------+---------+------------+--------------------+-------+--------+--------+----------+
|    28|US-2015-150630| 9/17/2015| 9/21/2015|Standard Class|   TB-21520|   Tracy Blumstein|   Consumer|United States|  Philadelphia|  Pennsylvania|      19140|   East|FUR-BO-10004834|Furniture|   Bookcases

Changing required string columns to double


In [11]:
from pyspark.sql.functions import col

# Convert Quantity from string to integer
df = df.withColumn(
    "Quantity",
    col("Quantity").cast("integer")
)

# Convert Discount from string to double
df = df.withColumn(
    "Discount",
    col("Discount").cast("double")
)

# Verify schema after conversion
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = false)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [12]:
high_sales = df.filter(col("Sales") > 500)
high_sales.show()


+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+--------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|          City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+--------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|    28|US-2015-150630| 9/17/2015| 9/21/2015|Standard Class|   TB-21520|   Tracy Blumstein|   Consumer|United States|  Philadelphia|  Pennsylvania|      19140|   East|FUR-BO-10004834| 

#Step 6: Transformation

In [13]:
df = df.withColumnRenamed(
    "Sales",
    "Total_Sales"
)

type Casting Quantity

In [14]:
df = df.withColumn(
    "Quantity",
    col("Quantity").cast("integer")
)

df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = false)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Total_Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



#Step 7 : Aggregations

In [15]:
print("Total Orders:", df.count())

Total Orders: 9994


Average Sales

In [16]:
df.select(
    avg("Total_Sales")
).show()

+-----------------+
| avg(Total_Sales)|
+-----------------+
|229.8580008304968|
+-----------------+



Maximum Sales

In [17]:
df.select(
    max("Total_Sales")
).show()

+----------------+
|max(Total_Sales)|
+----------------+
|        22638.48|
+----------------+



Minimum Sales

In [18]:
df.select(
    min("Total_Sales")
).show()

+----------------+
|min(Total_Sales)|
+----------------+
|           0.444|
+----------------+



Total sales

In [19]:
df.select(
    sum("Profit")
).show()

+-----------------+
|      sum(Profit)|
+-----------------+
|286397.0216999997|
+-----------------+



#Step 8 : Grouby By

Sales by Category

In [20]:
category_sales = df.groupBy(
    "Category"
).agg(
    sum("Total_Sales").alias("Total_Sales")
)

category_sales.show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies| 719047.032000001|
|      Furniture|741999.7953000009|
|     Technology|836154.0329999996|
+---------------+-----------------+



Profit By region

In [21]:
region_profit = df.groupBy(
    "Region"
).agg(
    sum("Profit").alias("Total_Profit")
)

region_profit.show()

+-------+-----------------+
| Region|     Total_Profit|
+-------+-----------------+
|  South|46749.43030000002|
|Central|       39706.3625|
|   East|         91522.78|
|   West|      108418.4489|
+-------+-----------------+



Average Sales By segment

In [22]:
segment_sales = df.groupBy(
    "Segment"
).agg(
    avg("Total_Sales").alias("Average_Sales")
)

segment_sales.show()

+-----------+------------------+
|    Segment|     Average_Sales|
+-----------+------------------+
|   Consumer|223.73364380658845|
|Home Office|240.97204066180595|
|  Corporate|233.82330026490058|
+-----------+------------------+



Aggregated condition

In [23]:
category_sales.filter(
    col("Total_Sales") > 700000
).show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies| 719047.032000001|
|      Furniture|741999.7953000009|
|     Technology|836154.0329999996|
+---------------+-----------------+



#Step 9 : Wide transformation example below

In [24]:
df.groupBy("Region").count().show()

+-------+-----+
| Region|count|
+-------+-----+
|  South| 1620|
|Central| 2323|
|   East| 2848|
|   West| 3203|
+-------+-----+



groupBy() is a wide transformation because spark must move data between partitions

this movement is called shuffle

shuffle is expensive because data travels across the cluster

#Step 10 : complete Pipeline

In [27]:
pipeline_df = (
    spark.read.csv(
        "/content/Main_dataset/Sample - Superstore.csv",
        header=True,
        inferSchema=True,
        quote='"',
        escape='"',
        multiLine=True
    )
    .dropDuplicates()
    .fillna({"Postal Code": 0})
    .filter(col("Sales") > 100)
    .withColumnRenamed("Sales", "Total_Sales")
)

In [28]:
result = pipeline_df.groupBy(
    "Category"
).agg(
    sum("Total_Sales").alias("Category_Sales"),
    avg("Profit").alias("Average_Profit")
)

result.show()

+---------------+-----------------+------------------+
|       Category|   Category_Sales|    Average_Profit|
+---------------+-----------------+------------------+
|Office Supplies| 587882.706999999| 77.08433658730158|
|      Furniture|710789.0128999997|12.078401365705613|
|     Technology|802689.8799999984| 117.9498173546756|
+---------------+-----------------+------------------+



In [30]:
from pyspark.sql.types import DoubleType
import glob, shutil, os

# small helper - spark always writes a folder of part files, never one clean csv
# so we coalesce to 1 partition, grab that single part file, rename it, and clean up
def save_single_csv(spark_df, temp_folder, final_name):
    spark_df.coalesce(1).write.mode("overwrite") \
        .option("header", True) \
        .csv(temp_folder)

    part_file = glob.glob(f"{temp_folder}/part-*.csv")[0]
    shutil.move(part_file, final_name)
    shutil.rmtree(temp_folder)   # remove the temp folder, keep just the renamed file


# make sure the output folder exists
os.makedirs("/content/Main_dataset/Output", exist_ok=True)

# 1. save the cleaned + filtered dataset (row-level data)
save_single_csv(
    pipeline_df,
    "/content/main_dataset/Output/cleaned_temp",
    "/content/Main_dataset/Output/cleaned_data.csv"
)

# 2. save the groupBy summary (aggregated insights)
save_single_csv(
    result,
    "/content/Main_dataset/Output/summary_temp",
    "/content/Main_dataset/Output/category_summary.csv"
)

print("Saved both files to /content/Main_dataset/Output")

Saved both files to /content/Main_dataset/Output
